# Part 3 — Ranking & Filtering

| Method | Status |
|---|---|
| 1a. TF-IDF + cosine similarity | done |
| 1b | to do |
| 1c | to do |

Rankings are registered in `RANKINGS_LIST` (`myapp/search/algorithms.py`) and selected with `search_in_corpus(query, index, ranking=...)`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "myapp").exists())
sys.path.insert(0, str(ROOT))

## 1. Load the index and the corpus

In [2]:
import pandas as pd
from myapp.search.indexing import load_index
from myapp.search.algorithms import process_query, conjunctive_retrieve, search_in_corpus, RANKINGS_LIST
from myapp.search.evaluation import (labels_in_rank_order, count_relevant, precision_at_k,
                                     average_precision, average_precision_at_k, mean_average_precision,
                                     reciprocal_rank, mean_reciprocal_rank, ndcg_at_k)

index = load_index(str(ROOT / "data" / "inverted_index.pkl"))
processed = pd.read_pickle(ROOT / "data" / "processed_corpus.pkl")
titles = processed.set_index("pid")["title"]
print(f"documents: {index.n_docs} | vocabulary: {len(index.vocab)} | rankings: {sorted(RANKINGS_LIST)}")

documents: 28080 | vocabulary: 5053 | rankings: ['tfidf']


## 2. Test queries

The five queries of Part 2, with the number of documents matched by the AND retrieval.

In [3]:
queries = {
    "q1": "men round neck cotton tshirt",
    "q2": "women casual cotton blend topwear",
    "q3": "men slim fit blue jeans",
    "q4": "black solid cotton polo",
    "q5": "women full sleeve printed sweatshirt",
}
pd.DataFrame([{"query": qid, "text": q, "terms": process_query(q),
               "AND matches": len(conjunctive_retrieve(process_query(q), index))}
              for qid, q in queries.items()])

,query,text,terms,AND matches
0,q1,men round neck cotton tshirt,"[men, round, neck, cotton, tshirt]",3678
1,q2,women casual cotton blend topwear,"[women, casual, cotton, blend, topwear]",1548
2,q3,men slim fit blue jeans,"[men, slim, fit, blue, jean]",104
3,q4,black solid cotton polo,"[black, solid, cotton, polo]",222
4,q5,women full sleeve printed sweatshirt,"[women, full, sleev, print, sweatshirt]",341


## 3. Ranking 1a: TF-IDF + cosine similarity

Query and documents are vectors over the vocabulary with weights `(1 + log tf) · log(N / df)`. A document's score is the cosine of the angle between its vector and the query vector.

In [8]:
def show(query, k=10, ranking="tfidf"):
    results = search_in_corpus(query, index, top_k=k, ranking=ranking)
    print(f"Query: {query!r} -> terms {process_query(query)} | ranking: {ranking}")
    return pd.DataFrame([(pid, titles[pid], round(score, 4)) for pid, score in results],
                        columns=["pid", "title", "score"])

show(queries["q1"], k=5)

Query: 'men round neck cotton tshirt' -> terms ['men', 'round', 'neck', 'cotton', 'tshirt'] | ranking: tfidf


,pid,title,score
0,TSHFKJESD3ZAESPF,Solid Men Round Neck Black T-Shirt,0.7492
1,TSHFUSKPWUZSP6F6,Printed Men Round Neck Black T-Shirt,0.7233
2,TSHFKJEVTH5ZYDAB,Solid Men Round Neck Black T-Shirt,0.7205
3,TSHFKJF5XCHHHZAV,Solid Men Round Neck Red T-Shirt,0.6393
4,TSHFKAJTQMPRVPKD,Solid Men Round Neck Dark Blue T-Shirt,0.6321


## 4. Evaluation

The judgments are the labels of Part 2 (`test_queries_labels.csv`, top 20 of the TF-IDF ranking per query). Only judged documents count, as in Part 2, so the table is comparable across methods.

In [9]:
labels_df = pd.read_csv(ROOT / "project_progress" / "part_2" / "test_queries_labels.csv")
judgments = {qid: dict(zip(g["pid"], g["label"])) for qid, g in labels_df.groupby("query_id")}
KS = (5, 10, 20)

def evaluate(ranking):
    """Per-query and global metrics of one ranking method over the test queries."""
    rows = []
    for qid, q in queries.items():
        ranked = [pid for pid, _ in search_in_corpus(q, index, ranking=ranking)]
        judged = judgments[qid]
        labels = labels_in_rank_order(ranked, judged)
        n_rel = count_relevant(list(judged.values()))
        row = {"ranking": ranking, "query": qid, "AP": average_precision(labels, n_rel), "RR": reciprocal_rank(labels)}
        for k in KS:
            row[f"P@{k}"] = precision_at_k(labels, k)
            row[f"AP@{k}"] = average_precision_at_k(labels, k, n_rel)
            row[f"NDCG@{k}"] = ndcg_at_k(labels, k, list(judged.values()))
        rows.append(row)
    return pd.DataFrame(rows)

per_query = evaluate("tfidf")
per_query.round(3)

,ranking,query,AP,RR,P@5,AP@5,NDCG@5,P@10,AP@10,NDCG@10,P@20,AP@20,NDCG@20
0,tfidf,q1,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
1,tfidf,q2,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
2,tfidf,q3,0.771,1.0,0.8,0.76,0.854,0.7,0.604,0.763,0.55,0.771,0.924
3,tfidf,q4,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000
4,tfidf,q5,1.000,1.0,1.0,1.00,1.000,1.0,1.000,1.000,1.00,1.000,1.000


In [10]:
def summary(per_query_df):
    return pd.DataFrame([{"ranking": r, "MAP": mean_average_precision(g["AP"].tolist()),
                          "MRR": mean_reciprocal_rank(g["RR"].tolist()), "mean NDCG@10": g["NDCG@10"].mean()}
                         for r, g in per_query_df.groupby("ranking", sort=False)]).round(3)

summary(per_query)

,ranking,MAP,MRR,mean NDCG@10
0,tfidf,0.954,1.0,0.953
